In [95]:
from spin_lattices import SpinLattice, KagomeLattice, SquareLattice, TriangleLattice
import numpy.typing as npt
import pandas as pd
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path

In [46]:
class AllToAllLattice(SpinLattice):
    def __init__(self, original_lattice: SpinLattice):
        super().__init__()
        self.original_lattice = original_lattice
        self.lattice_basis = original_lattice.lattice_basis
        self.site_to_num = original_lattice.site_to_num
        sites = lattice.sites_df.query("is_canonical")[["ix", "iy"]].to_numpy()
        self.edges: list[tuple[tuple[npt.NDArray, npt.NDArray], int]] = [  # (start, end), kind
            ((p, q), 1) for p in sites for q in sites
        ]

    @property
    def sites_df(self) -> pd.DataFrame:
        return self.original_lattice.sites_df
    
    def get_cache_id(self) -> str:
        return f"AllToAll_{self.original_lattice.get_cache_id()}"
        

2023-08-07 20:56:54.800 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=18
2023-08-07 20:56:54.803 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-07 20:56:54.819 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 48620
2023-08-07 20:56:54.829 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:67 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x3-1.0-0.9-False-None-1.pickle
2023-08-07 20:56:54.831 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:114 - Ground state energy is -31.2798921416


(array([-31.27989214]),
 array([[ 9.72687223e-06],
        [ 2.92635025e-05],
        [-6.36622588e-05],
        ...,
        [ 6.36622588e-05],
        [-2.92635025e-05],
        [-9.72687223e-06]]))

In [98]:
lattice = KagomeLattice(2, 2)

full_spin_values = []
for J2 in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]:
    system = HeisenbergJ1J2(
        lattice,
        J1=1,
        J2=J2,
        use_symmetries=False,
        spin_inversion=None,
        ground_state_cache_dir=Path("groundstates"),
    )
    system.get_eigenstates(1)

    full_spin_system = HeisenbergJ1J2(
        lattice=AllToAllLattice(lattice),
        J1=1,
        use_symmetries=False,
        spin_inversion=None,
    )
    full_spin_values.append(
        {
            "J2": J2,
            "full_spin": system.get_ground_state_in_canonical_basis()
            @ (full_spin_system.hamiltonian @ system.get_ground_state_in_canonical_basis()),
        }
    )

2023-08-07 21:03:46.582 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=12
2023-08-07 21:03:46.583 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-07 21:03:46.595 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 924
2023-08-07 21:03:46.601 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:67 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x2-1.0-0.1-False-None-1.pickle
2023-08-07 21:03:46.614 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:114 - Ground state energy is -22.6063143323
2023-08-07 21:03:46.621 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=12
2023-08-07 21:03:46.622 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-07 21:03:46.633 | DE

[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
2023-08-07 21:03:46.714 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=12
2023-08-07 21:03:46.716 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
2023-08-07 21:03:46.717 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 924
2023-08-07 21:03:46.723 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:67 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x2-1.0-0.2-False-None-1.pickle
2023-08-07 21:03:46.726 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:114 - Ground state energy is -21.9239567839
2023-08-07 21:03:46.734 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=12
2023-08-07 21:03:46.735 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives 

In [99]:
full_spin_values

[{'J2': 0.1, 'full_spin': 23.999999999999993},
 {'J2': 0.2, 'full_spin': 23.999999999999943},
 {'J2': 0.3, 'full_spin': 24.000000000000014},
 {'J2': 0.4, 'full_spin': 23.99999999999997},
 {'J2': 0.5, 'full_spin': 24.000000000000025},
 {'J2': 0.6, 'full_spin': 9.41961092050124e-16},
 {'J2': 0.7, 'full_spin': 8.525441619561936e-16},
 {'J2': 0.8, 'full_spin': -9.18807533926755e-16},
 {'J2': 0.9, 'full_spin': 3.709667855811176e-16},
 {'J2': 1, 'full_spin': -1.9730456382444543e-16}]

In [101]:
system = HeisenbergJ1J2(
    lattice,
    J1=1,
    J2=0.3,
    use_symmetries=False,
    spin_inversion=None,
    ground_state_cache_dir=Path("groundstates"),
)
system.get_eigenstates(2)

2023-08-07 21:04:48.249 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=12
2023-08-07 21:04:48.253 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
2023-08-07 21:04:48.255 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 924
2023-08-07 21:04:48.268 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:101 - Calculating eigenvalues / eigenstates
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 160; batchedOperatorChunkSize: 6
[Debug]   [LOCALE

(array([-21.25110689, -20.05004008]),
 array([[ 0.01440412,  0.02098492],
        [-0.01837589, -0.01334276],
        [ 0.01440412,  0.01763069],
        ...,
        [ 0.01440412, -0.01763069],
        [-0.01837589,  0.01334276],
        [ 0.01440412, -0.02098492]]))